In [2]:
"""
Glass Brain Render — Superior View with Reflection
===================================================
Produces a translucent blue glass-brain figure in the style of Hagmann et al.
showing an Adult, Child, and Macaque brain side-by-side.

Requirements:
    pip install templateflow nibabel scikit-image scipy matplotlib pyvista

For off-screen rendering (servers / headless):
    pip install pyvistaqt   OR   set PYVISTA_OFF_SCREEN=true before running
    On Linux without a display:  apt-get install libgl1-mesa-glx xvfb
                                 xvfb-run python glass_brain_render.py
"""

import os
os.environ["PYVISTA_OFF_SCREEN"] = "true"   # remove for interactive window

import numpy as np
import pyvista as pv
from skimage import measure
from scipy.ndimage import gaussian_filter, zoom
import nibabel as nib
import templateflow.api as tflow

# ─── colour palette ──────────────────────────────────────────────────────────
BRAIN_BLUE    = (0.45, 0.62, 0.82)   # steel-blue glass
BG_WHITE      = (1.0,  1.0,  1.0)
REFLECTION_Y  = -1.10   # how far below (fraction of bounding-box height)

# ─── templateflow fetch ──────────────────────────────────────────────────────
print("Fetching templates …")

adult_t1 = tflow.get('MNI152NLin2009cAsym', resolution=2,
                     desc='brain', suffix='T1w', extension='.nii.gz')
child_t1 = tflow.get('MNIPediatricAsym', cohort='2', resolution=2,
                     suffix='T1w', extension='.nii.gz')
monkey_t1 = tflow.get('NMT31Sym', resolution=1,
                      desc='brain', suffix='T1w', extension='.nii.gz')


Fetching templates …


100%|██████████| 452k/452k [00:00<00:00, 1.40MB/s]
100%|██████████| 1.96M/1.96M [00:00<00:00, 3.71MB/s]
100%|██████████| 9.67M/9.67M [00:01<00:00, 5.37MB/s]


In [12]:
# ─── load + normalise ────────────────────────────────────────────────────────

def load_norm(path, smooth=1.2):
    img  = nib.load(str(path))
    img  = nib.as_closest_canonical(img)
    data = img.get_fdata(dtype=np.float32)
    # data = gaussian_filter(data, sigma=smooth)
    zooms = np.array(img.header.get_zooms()[:3], dtype=float)
    # normalise to [0, 1]
    mn, mx = data[data > 0].min(), data.max()
    data = np.clip((data - mn) / (mx - mn), 0, 1)
    return data, zooms

print("Loading images …")
adult_vol,  adult_z  = load_norm(adult_t1,  smooth=1.2)
child_vol,  child_z  = load_norm(child_t1,  smooth=1.8)   # smoother child gyri
monkey_vol, monkey_z = load_norm(monkey_t1, smooth=1.0)


Loading images …


In [26]:

# ─── isosurface extraction ───────────────────────────────────────────────────
# Stack several thresholds → overlapping translucent meshes recreate the
# glass-brain appearance (outer cortex + WM boundary + subcortical structures)

THRESHOLDS_HUMAN  = [0.15, 0.28, 0.42, 0.58]
THRESHOLDS_MONKEY = [0.15, 0.28, 0.42, 0.60]

def extract_meshes(volume, zooms, thresholds, step=2):
    meshes = []
    for thr in thresholds:
        try:
            verts, faces, _, _ = measure.marching_cubes(
                volume, level=thr, step_size=step, spacing=zooms
            )
            # Build PyVista PolyData
            n = len(faces)
            cells = np.hstack([np.full((n, 1), 3), faces]).ravel()
            mesh = pv.PolyData(verts, cells)
            mesh = mesh.clean().smooth(n_iter=30, relaxation_factor=0.1)
            meshes.append(mesh)
        except Exception:
            pass
    return meshes

print("Extracting surfaces …")
adult_meshes  = extract_meshes(adult_vol,  adult_z,  THRESHOLDS_HUMAN,  step=2)
# adult_meshes = adult_meshes[1]  # keep only outer surfaces for adult (too many details)
child_meshes  = extract_meshes(child_vol,  child_z,  THRESHOLDS_HUMAN,  step=2)
monkey_meshes = extract_meshes(monkey_vol, monkey_z, THRESHOLDS_MONKEY, step=1)

# ─── per-threshold opacity (outer surface = barely-there; inner = more solid) 
OPACITIES_HUMAN  = [0, 0, 0.10, 0]
OPACITIES_MONKEY = [0, 0, 0.10, 0]


Extracting surfaces …


In [27]:
adult_meshes

[PolyData (0x1415b3e20)
   N Cells:    16124
   N Points:   8076
   N Strips:   0
   X Bounds:   2.412e+01, 1.683e+02
   Y Bounds:   2.493e+01, 2.064e+02
   Z Bounds:   6.739e+00, 1.605e+02
   N Arrays:   0,
 PolyData (0x14149c400)
   N Cells:    18044
   N Points:   9086
   N Strips:   0
   X Bounds:   2.556e+01, 1.676e+02
   Y Bounds:   2.561e+01, 2.056e+02
   Z Bounds:   7.230e+00, 1.600e+02
   N Arrays:   0,
 PolyData (0x14149e680)
   N Cells:    22886
   N Points:   11563
   N Strips:   0
   X Bounds:   2.651e+01, 1.671e+02
   Y Bounds:   2.632e+01, 2.047e+02
   Z Bounds:   7.597e+00, 1.595e+02
   N Arrays:   0,
 PolyData (0x14149e6e0)
   N Cells:    37024
   N Points:   18656
   N Strips:   0
   X Bounds:   2.760e+01, 1.648e+02
   Y Bounds:   2.712e+01, 2.037e+02
   Z Bounds:   7.981e+00, 1.590e+02
   N Arrays:   0]

In [28]:

# ─── helper: centre + normalise scale ────────────────────────────────────────

def centre(meshes):
    """Translate all meshes so the outermost one is centred at origin."""
    if not meshes:
        return meshes
    pts = meshes[-1].points
    ctr = pts.mean(axis=0)
    return [m.translate(-ctr, inplace=False) for m in meshes]

adult_meshes  = centre(adult_meshes)
child_meshes  = centre(child_meshes)
monkey_meshes = centre(monkey_meshes)


# ─── compute x-offsets so the three brains sit side-by-side ──────────────────

def x_extent(meshes):
    if not meshes:
        return 100
    b = meshes[-1].bounds   # xmin, xmax, ...
    return b[1] - b[0]

spacing = 30   # mm gap between brains
w_adult  = x_extent(adult_meshes)
w_child  = x_extent(child_meshes)
w_monkey = x_extent(monkey_meshes)

# centre each panel
x_adult  = 0
x_child  = -(w_adult/2 + spacing + w_child/2)
x_monkey =  (w_adult/2 + spacing + w_monkey/2)

def shift_x(meshes, dx):
    return [m.translate([dx, 0, 0], inplace=False) for m in meshes]

adult_meshes  = shift_x(adult_meshes,  x_adult)
child_meshes  = shift_x(child_meshes,  x_child)
monkey_meshes = shift_x(monkey_meshes, x_monkey)


In [29]:


# ─── render ──────────────────────────────────────────────────────────────────
print("Rendering …")

pl = pv.Plotter(off_screen=True, window_size=(2400, 1200))
pl.set_background(BG_WHITE)

def add_brain(plotter, meshes, opacities, x_shift, color, refl_factor=0.45):
    """Add a brain (and its reflection) to the plotter."""
    for mesh, op in zip(meshes, opacities):
        plotter.add_mesh(
            mesh,
            color=color,
            opacity=op,
            smooth_shading=True,
            specular=0.6,
            specular_power=50,
            ambient=0.3,
            diffuse=0.7,
        )

def add_reflection(plotter, meshes, opacities, color, refl_factor=0.40):
    for mesh, op in zip(meshes, opacities):
        plotter.add_mesh(
            mesh,
            color=color,
            opacity=op * refl_factor,
            smooth_shading=True,
            specular=0.2,
            ambient=0.2,
            diffuse=0.5,
        )

add_brain(pl, adult_meshes,  OPACITIES_HUMAN,  x_adult,  BRAIN_BLUE)
add_brain(pl, child_meshes,  OPACITIES_HUMAN,  x_child,  BRAIN_BLUE)
add_brain(pl, monkey_meshes, OPACITIES_MONKEY, x_monkey, BRAIN_BLUE)

# add_reflection(pl, adult_refl,  OPACITIES_HUMAN,  BRAIN_BLUE)
# add_reflection(pl, child_refl,  OPACITIES_HUMAN,  BRAIN_BLUE)
# add_reflection(pl, monkey_refl, OPACITIES_MONKEY, BRAIN_BLUE)

# ─── camera: straight down (superior view) ───────────────────────────────────
# Position camera far above, looking straight down; roll so anterior = up
pl.camera_position = 'xy'                    # look along -Z
pl.camera.elevation = 90
pl.camera.azimuth   = 0
# Fine-tune: pull camera above and look down at Y=0 plane
pl.camera.position  = (0, 0, 600)
pl.camera.focal_point = (0, 0, 0)
pl.camera.up        = (0, 1, 0)
pl.camera.zoom(0.9)

# ─── labels ──────────────────────────────────────────────────────────────────
# font_args = dict(font_size=22, color='#4a6080', bold=True)

if adult_meshes:
    pl.add_text("Adult Human",    position=(0.19, 0.90), viewport=True) # , **font_args)
    pl.add_text("MNI152NLin2009cAsym", position=(0.12, 0.86), viewport=True,
                font_size=13, color='#8090a0')

if child_meshes:
    pl.add_text("Child (~6 yr)",  position=(0.49, 0.90), viewport=True) # , **font_args)
    pl.add_text("MNIPediatricAsym coh-2", position=(0.43, 0.86), viewport=True,
                font_size=13, color='#8090a0')

if monkey_meshes:
    pl.add_text("Rhesus Macaque", position=(0.77, 0.90), viewport=True) # , **font_args)
    pl.add_text("NMT31Sym",       position=(0.81, 0.86), viewport=True,
                # font_size=13, 
                # color='#8090a0'
                )

pl.add_text(
    "Brain Morphology Across Species & Development  ·  Superior View",
    position=(0.5, 0.96), viewport=True, font_size=18,
    color='#2a3a50', # bold=True,
)
# pl.add_text(
#     "templateflow.org",
#     position=(0.5, 0.02), viewport=True, font_size=11, color='#aabbcc',
# )

out = "/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/output/brains_from_above/glass_brain_render.png"
pl.screenshot(out, transparent_background=True)
pl.close()
print(f"\nSaved → {out}")

Rendering …

Saved → /Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/output/brains_from_above/glass_brain_render.png


# OLD


In [2]:

# ── 1. Fetch masks via templateflow ─────────────────────────────────────────
#    First call downloads to ~/.cache/templateflow; subsequent calls are instant.

print("Fetching Adult (MNI152NLin2009cAsym) …")
adult_path = tflow.get(
    'MNI152NLin2009cAsym',
    resolution=2,           # 2 mm — faster marching cubes
    desc='brain',
    suffix='mask',
    extension='.nii.gz',
)

print("Fetching Child (MNIPediatricAsym, cohort-2, ~4.5–8.5 yr) …")
child_path = tflow.get(
    'MNIPediatricAsym',
    cohort='2',
    resolution=2,
    desc='brain',
    suffix='mask',
    extension='.nii.gz',
)

print("Fetching Macaque (NMT31Sym) …")
monkey_path = tflow.get(
    'NMT31Sym',
    resolution=1,           # only res-1 available
    desc='brain',
    suffix='mask',
    extension='.nii.gz',
)


Fetching Adult (MNI152NLin2009cAsym) …
Fetching Child (MNIPediatricAsym, cohort-2, ~4.5–8.5 yr) …
Fetching Macaque (NMT31Sym) …


In [3]:

# ── 2. Load, reorient to RAS, light smooth ──────────────────────────────────

def load_brain(path, smooth_sigma=1.5):
    img = nib.load(str(path))
    img = nib.as_closest_canonical(img)     # → RAS orientation
    data = img.get_fdata(dtype=np.float32)
    data = gaussian_filter(data, sigma=smooth_sigma)
    return data

adult_data  = load_brain(adult_path,  smooth_sigma=1.5)
child_data  = load_brain(child_path,  smooth_sigma=2.0)   # slightly smoother sulci
monkey_data = load_brain(monkey_path, smooth_sigma=1.5)


In [4]:

# ── 3. Marching-cubes surface extraction ────────────────────────────────────

def get_mesh(volume, threshold=0.35, step=2):
    verts, faces, _, _ = measure.marching_cubes(
        volume, level=threshold, step_size=step
    )
    # Scale verts by voxel size so physical mm are comparable across templates
    verts -= verts.mean(axis=0)
    return verts, faces

print("Extracting surfaces …")
av, af = get_mesh(adult_data,  step=2)
cv, cf = get_mesh(child_data,  step=2)
mv, mf = get_mesh(monkey_data, step=1)   # smaller volume → finer step


Extracting surfaces …


In [ ]:

# ── 4. Figure ────────────────────────────────────────────────────────────────

GREY   = '#C8C8CC'
BG     = '#0E0E14'
TITLE_C = '#DDDDE8'
SUB_C   = '#9090AA'

fig = plt.figure(figsize=(16, 7), facecolor=BG)
fig.patch.set_facecolor(BG)

panels = [
    (av, af, 'Adult Human',    'MNI152NLin2009cAsym\n(res-2 mm)',          '~1 350 cc', 1),
    (cv, cf, 'Child (~6 yr)',  'MNIPediatricAsym cohort-2\n(4.5–8.5 yr)',   '~1 200 cc', 2),
    (mv, mf, 'Rhesus Macaque', 'NMT31Sym\n(res-1 mm)',                      '~85 cc',    3),
]

for verts, faces, title, subtitle, vol, pos in panels:
    ax = fig.add_subplot(1, 3, pos, projection='3d', facecolor=BG)

    poly = Poly3DCollection(
        verts[faces],
        alpha=0.55,
        linewidths=0,
        facecolor=GREY,
        edgecolors='none',
    )
    ax.add_collection3d(poly)

    pad = 2
    ax.set_xlim(verts[:, 0].min()-pad, verts[:, 0].max()+pad)
    ax.set_ylim(verts[:, 1].min()-pad, verts[:, 1].max()+pad)
    ax.set_zlim(verts[:, 2].min()-pad, verts[:, 2].max()+pad)

    ax.view_init(elev=90, azim=-90)   # straight down = superior view
    ax.set_axis_off()
    ax.set_facecolor(BG)

    ax.set_title(title, color=TITLE_C, fontsize=16, fontweight='bold', pad=8)
    fig.text(
        (pos - 0.5) / 3, 0.04,
        subtitle,
        ha='center', va='bottom',
        color=SUB_C, fontsize=8.5, style='italic',
        transform=fig.transFigure,
    )
    ax.text2D(0.96, 0.96, vol, transform=ax.transAxes,
              ha='right', va='top', color=SUB_C, fontsize=8)

fig.suptitle(
    'Brain Morphology Across Species & Development  ·  Superior View',
    color=TITLE_C, fontsize=14, fontweight='bold', y=0.97,
)
fig.text(
    0.5, 0.01,
    'Data: templateflow.org  |  Adult: MNI152NLin2009cAsym  |  '
    'Child: MNIPediatricAsym  |  Macaque: NMT31Sym',
    ha='center', va='bottom', color='#555566', fontsize=7,
    transform=fig.transFigure,
)

plt.tight_layout(rect=[0, 0.09, 1, 0.95])

out = 'brain_comparison.pdf'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor=BG)
print(f"\nSaved → {out}")
plt.show()

In [6]:
plt.show()

/var/folders/b2/80yhhgwn2fjb5qv0cbj_cr6c0000gn/T/ipykernel_51324/2122436968.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
